In [14]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import io
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.neural_network import MLPClassifier
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
PROJECT_ROOT  = Path.cwd().parent
DATA_PATH     = PROJECT_ROOT / "data" / "processed" / "train.parquet"
MODEL_PATH    = PROJECT_ROOT / "models" / "sklearn"

con = duckdb.connect(database=":memory:")
con.execute(f"CREATE VIEW train AS SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')")

con.sql("DESCRIBE train").df()

,column_name,column_type,null,key,default,extra
0,click,TINYINT,YES,None,None,None
1,day,INTEGER,YES,None,None,None
2,hour_sin,DOUBLE,YES,None,None,None
3,hour_cos,DOUBLE,YES,None,None,None
4,time_band,VARCHAR,YES,None,None,None
5,log_freq_device_id,DOUBLE,YES,None,None,None
6,log_freq_device_ip,DOUBLE,YES,None,None,None
7,site_id,VARCHAR,YES,None,None,None
8,site_domain,VARCHAR,YES,None,None,None
9,site_category,VARCHAR,YES,None,None,None


In [2]:
def get_stratified_sample(
    con,
    relation,
    n=1_000_000,
    target="click",
    seed=42,
):
    """Draw a stratified sample of ``n`` rows from a DuckDB relation.

    The sample preserves the proportion of the target classes of the
    source relation. The number of rows drawn per class is proportional
    to its frequency in the source, and the sampling within each class is
    reproducible given the same seed.

    Args:
        con (duckdb.DuckDBPyConnection): Active DuckDB connection.
        relation (str): Name of a table or view.
        n (int): Total number of rows to draw. Defaults to 1,000,000.
        target (str): Column whose distribution must be preserved.
            Defaults to ``"click"``.
        seed (int): Seed for the reservoir sampler. Defaults to 42.

    Returns:
        pandas.DataFrame: The sampled rows.

    Raises:
        TypeError: If ``con`` is not a DuckDB connection, or if
            ``relation`` or ``target`` is not a string.
        ValueError: If ``n`` is not a positive integer, or if ``n`` is
            not strictly smaller than the relation size.
    """
    import duckdb

    if not isinstance(con, duckdb.DuckDBPyConnection):
        raise TypeError("'con' must be a DuckDB connection.")

    if not isinstance(relation, str):
        raise TypeError("'relation' must be a string.")

    if not isinstance(target, str):
        raise TypeError("'target' must be a string.")

    if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
        raise ValueError("'n' must be a positive integer.")

    total = int(
        con.sql(f"SELECT COUNT(*) AS n FROM {relation}").df()["n"].iloc[0]
    )

    if n >= total:
        raise ValueError(
            f"'n' ({n:,}) must be smaller than the relation size "
            f"({total:,})."
        )

    df_classes = con.sql(f"""
        SELECT "{target}" AS cls, COUNT(*) AS cnt
        FROM {relation}
        GROUP BY "{target}"
    """).df()

    parts = []
    for _, row in df_classes.iterrows():
        cls = row["cls"]
        cnt = int(row["cnt"])
        n_cls = max(1, round(n * cnt / total))
        parts.append(f"""
            SELECT * FROM (
                SELECT * FROM {relation}
                WHERE "{target}" = {cls}
            ) AS filtered
            USING SAMPLE {n_cls} ROWS (reservoir, {seed})
        """)

    query = " UNION ALL ".join(parts)
    return con.sql(query).df()

In [18]:
"""Custom preprocessing transformer for the Avazu CTR dataset."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder


DEFAULT_TOP_K = {
    "banner_pos":    2,
    "C18":           3,
    "app_category":  3,
    "site_category": 3,
    "app_domain":    6,
    "C19":          10,
    "C20":          10,
    "C21":          10,
}


class AvazuPreprocessor(BaseEstimator, TransformerMixin):
    """Apply the encoding strategy defined in the EDA to raw Avazu columns.

    The transformer learns all state-dependent statistics (dominant
    category, top-K ranking, target rate per category) from the training
    data during ``fit`` and applies them to any subsequent data during
    ``transform``. It never touches the target of the data passed to
    ``transform``, so it can be safely applied to the test set.

    Encoding strategies applied:
        - Dichotomization: replace each column by a binary indicator of
          its dominant category.
        - Top-K + Other: keep the K most frequent categories and collapse
          the rest into a single ``Other`` level.
        - Hybrid: emit a binary indicator of the dominant category and a
          target-encoded column for the rest.
        - Target encoding: replace each category by its smoothed target
          rate.
        - One-hot: encode the time band and the top-K columns with
          ``OneHotEncoder``.

    Args:
        dichotomize (list of str): Columns to dichotomize.
        top_k (dict or None): Mapping from column name to the number of
            top categories to keep. When ``None``, ``DEFAULT_TOP_K`` is
            used. Defaults to ``None``.
        te_hybrid (list of str): Columns that receive the hybrid encoding
            (binary dominant + target encoding).
        te_pure (list of str): Columns that receive pure target encoding.
        numeric (list of str): Columns passed through unchanged.
        time_band (str): Name of the time band column, encoded with
            one-hot.
        smoothing (float): Smoothing parameter for target encoding.
            Defaults to 50.
        min_category_size (int): Categories with fewer than this many
            training occurrences are treated as ``Other`` for target
            encoding purposes. Defaults to 10.
    """

    def __init__(
        self,
        dichotomize=("C1", "C15", "C16", "device_type", "device_conn_type"),
        top_k=None,
        te_hybrid=("app_id", "site_domain", "site_id"),
        te_pure=("C14", "C17", "device_model"),
        numeric=(
            "day", "hour_sin", "hour_cos",
            "log_freq_device_id", "log_freq_device_ip",
        ),
        time_band="time_band",
        smoothing=50,
        min_category_size=10,
    ):
        self.dichotomize = dichotomize
        self.top_k = top_k
        self.te_hybrid = te_hybrid
        self.te_pure = te_pure
        self.numeric = numeric
        self.time_band = time_band
        self.smoothing = smoothing
        self.min_category_size = min_category_size

    # -----------------------------------------------------------------
    # fit
    # -----------------------------------------------------------------
    def fit(self, X, y):
        """Learn the encoding statistics from the training data.

        Args:
            X (pandas.DataFrame): Raw feature matrix.
            y (pandas.Series or numpy.ndarray): Binary target.

        Returns:
            self: The fitted transformer.
        """
        self.top_k_ = (
            self.top_k if self.top_k is not None else dict(DEFAULT_TOP_K)
        )

        y = pd.Series(np.asarray(y).ravel(), name="__target__")

        self.dominant_ = {
            col: X[col].value_counts().idxmax()
            for col in self.dichotomize
        }

        self.dominant_hybrid_ = {
            col: X[col].value_counts().idxmax()
            for col in self.te_hybrid
        }

        self.top_k_lists_ = {
            col: X[col].value_counts().head(k).index.tolist()
            for col, k in self.top_k_.items()
        }

        self.te_tables_ = self._build_te_tables(X, y)

        self.ohe_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ohe_cols = list(self.top_k_.keys()) + [self.time_band]
        self.ohe_.fit(self._prepare_ohe_input(X, ohe_cols))
        self.ohe_columns_ = ohe_cols

        self.feature_names_out_ = self._compute_feature_names()

        return self

    # -----------------------------------------------------------------
    # transform
    # -----------------------------------------------------------------
    def transform(self, X):
        """Apply the fitted encodings to new data.

        Args:
            X (pandas.DataFrame): Raw feature matrix.

        Returns:
            numpy.ndarray: The encoded feature matrix as float32.
        """
        parts = []

        # 1. Numeric passthrough
        parts.append(X[list(self.numeric)].to_numpy(dtype=np.float32))

        # 2. Dichotomized
        for col in self.dichotomize:
            dominant = self.dominant_[col]
            parts.append(
                (X[col] == dominant).to_numpy(dtype=np.float32).reshape(-1, 1)
            )

        # 3. Hybrid: binary dominant + target encoding of the rest
        for col in self.te_hybrid:
            dominant = self.dominant_hybrid_[col]
            is_dominant = (X[col] == dominant).to_numpy(dtype=np.float32)
            parts.append(is_dominant.reshape(-1, 1))

            te_col = self._apply_te(X, col, self.te_tables_[col])
            parts.append(te_col.reshape(-1, 1))

        # 4. Pure target encoding
        for col in self.te_pure:
            te_col = self._apply_te(X, col, self.te_tables_[col])
            parts.append(te_col.reshape(-1, 1))

        # 5. One-hot of top-K and time_band
        ohe_input = self._prepare_ohe_input(X, self.ohe_columns_)
        parts.append(self.ohe_.transform(ohe_input).astype(np.float32))

        return np.hstack(parts).astype(np.float32)

    # -----------------------------------------------------------------
    # internals
    # -----------------------------------------------------------------
    def _build_te_tables(self, X, y):
        """Compute the smoothed target rate per category for each TE column."""
        global_mean = y.mean()
        tables = {}

        for col in list(self.te_hybrid) + list(self.te_pure):
            df = pd.DataFrame({"cat": X[col].values, "y": y.values})
            agg = df.groupby("cat")["y"].agg(["sum", "count"])
            agg["rate"] = (
                agg["sum"] + self.smoothing * global_mean
            ) / (agg["count"] + self.smoothing)
            tables[col] = {
                "map": agg["rate"].to_dict(),
                "fallback": global_mean,
            }

        return tables

    def _apply_te(self, X, col, table):
        """Apply the fitted target encoding table to a column."""
        return (
            X[col]
            .map(table["map"])
            .fillna(table["fallback"])
            .to_numpy(dtype=np.float32)
        )

    def _prepare_ohe_input(self, X, cols):
        """Prepare the input DataFrame for the one-hot encoder."""
        out = pd.DataFrame(index=X.index)
        for col in cols:
            if col == self.time_band:
                out[col] = X[col].astype(str)
            else:
                top = self.top_k_lists_[col]
                out[col] = X[col].where(X[col].isin(top), "Other").astype(str)
        return out

    def _compute_feature_names(self):
        """Compute the output feature names for downstream use."""
        names = list(self.numeric)
        names += [f"{c}_is_dominant" for c in self.dichotomize]
        for c in self.te_hybrid:
            names.append(f"{c}_is_dominant")
            names.append(f"{c}_te")
        names += [f"{c}_te" for c in self.te_pure]
        names += list(self.ohe_.get_feature_names_out(self.ohe_columns_))
        return names

    def get_feature_names_out(self, input_features=None):
        """Return the output feature names.

        Args:
            input_features: Ignored. Present for scikit-learn API
                compatibility.

        Returns:
            numpy.ndarray: Array of feature names.
        """
        return np.asarray(self.feature_names_out_)

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin


class SelectiveScaler(BaseEstimator, TransformerMixin):
    """Scale only a subset of columns identified by their index.

    Wraps a ``StandardScaler`` and applies it only to the columns whose
    positions are listed in ``indices``. The remaining columns are passed
    through unchanged. Useful when the input is a NumPy array produced by
    an upstream transformer that does not preserve column names.

    Args:
        indices (tuple or list of int): 0-based positions of the columns
            to scale. Must be non-empty.
    """

    def __init__(self, indices=(3, 4)):
        self.indices = indices

    def fit(self, X, y=None):
        """Fit the internal scaler on the selected columns.

        Args:
            X (numpy.ndarray): Input array.
            y: Ignored. Present for API compatibility.

        Returns:
            self: The fitted transformer.
        """
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X[:, self.indices])
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        """Scale the selected columns and pass the rest through.

        Args:
            X (numpy.ndarray): Input array.

        Returns:
            numpy.ndarray: The transformed array with the same shape.
        """
        X_out = np.array(X, dtype=np.float32, copy=True)
        X_out[:, self.indices] = self.scaler_.transform(X[:, self.indices])
        return X_out

    def get_feature_names_out(self, input_features=None):
        """Return the feature names (passthrough).

        Args:
            input_features: Array of input feature names.

        Returns:
            numpy.ndarray: The input feature names unchanged.
        """
        if input_features is None:
            return np.asarray(
                [f"x{i}" for i in range(self.n_features_in_)]
            )
        return np.asarray(input_features)

In [6]:
def run_grid_search_manual_kfold(
    pipe,
    param_grid,
    X_train,
    y_train,
    model_path,
    cv=5,
    scoring="roc_auc",
    results_filename="grid_results.json",
    random_state=42,
):
    """Run a grid search with manual stratified K-fold and JSON checkpoints.

    Iterates over the Cartesian product of ``param_grid`` values. For each
    combination, runs a stratified K-fold cross-validation where each fold
    is used once for validation and the remaining K-1 folds for training.
    The results of every combination, including the AUC and fit time of
    each individual fold, are written to a JSON file as soon as the
    combination finishes, so the search can be interrupted and resumed
    without losing progress. A tqdm progress bar shows the state in real
    time.

    The function does not refit the best combination on the full training
    set. That step is left to the caller.

    Args:
        pipe (sklearn.pipeline.Pipeline): Pipeline to tune. Cloned for
            each fold, so the original is never modified.
        param_grid (dict): Mapping from parameter names (with the pipeline
            step prefix, e.g. ``"mlpclassifier__alpha"``) to lists of
            candidate values.
        X_train (pandas.DataFrame): Raw feature matrix for training.
        y_train (pandas.Series or numpy.ndarray): Binary target.
        model_path (pathlib.Path): Directory where the checkpoint JSON will
            be written. Created if missing.
        cv (int): Number of cross-validation folds. Defaults to 5.
        scoring (str): Metric to optimize. Only ``"roc_auc"`` is currently
            supported. Defaults to ``"roc_auc"``.
        results_filename (str): Name of the checkpoint JSON file. Defaults
            to ``"grid_results.json"``.
        random_state (int): Seed for the stratified K-fold splitter.
            Defaults to 42.

    Returns:
        pandas.DataFrame: Sorted results, one row per combination, with
        columns ``signature``, ``params``, ``fold_scores``,
        ``fold_times``, ``mean_auc``, ``std_auc``, ``mean_fit_time``,
        ``total_fit_time``.

    Raises:
        ValueError: If ``scoring`` is not ``"roc_auc"``.
    """
    # ------------------------------------------------------------------
    # Standard library imports kept local to avoid polluting the module
    # ------------------------------------------------------------------
    import itertools
    import json
    import time

    # ------------------------------------------------------------------
    # Third-party imports
    # ------------------------------------------------------------------
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold
    from tqdm.auto import tqdm

    # ------------------------------------------------------------------
    # Validate the scoring metric
    # ------------------------------------------------------------------
    if scoring != "roc_auc":
        raise ValueError(
            f"Only 'roc_auc' is supported, got '{scoring}'."
        )

    # ------------------------------------------------------------------
    # Prepare output directory and file paths
    # ------------------------------------------------------------------
    model_path = Path(model_path)
    model_path.mkdir(parents=True, exist_ok=True)

    results_path = model_path / results_filename

    # ------------------------------------------------------------------
    # Build the Cartesian product of all hyperparameter values
    # ------------------------------------------------------------------
    keys = list(param_grid.keys())
    combinations = [
        dict(zip(keys, combo))
        for combo in itertools.product(*param_grid.values())
    ]
    total = len(combinations)

    logger.info("Grid search: %d combinations, cv=%d", total, cv)

    # ------------------------------------------------------------------
    # Load previous checkpoint if it exists
    # ------------------------------------------------------------------
    if results_path.exists():
        with open(results_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        data = payload.get("results", [])
        done_signatures = {entry["signature"] for entry in data}
        logger.info(
            "Resuming from checkpoint: %d/%d already done",
            len(done_signatures), total,
        )
    else:
        data = []
        done_signatures = set()
        payload = {
            "metadata": {
                "grid_size": total,
                "cv": cv,
                "scoring": scoring,
                "total_wall_time": 0.0,
            },
            "results": data,
        }

    # ------------------------------------------------------------------
    # Build the K folds once so all combinations share the same splits
    # ------------------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=random_state,
    )
    splits = list(skf.split(X_train, y_train))

    # ------------------------------------------------------------------
    # Progress bar over combinations
    # ------------------------------------------------------------------
    bar = tqdm(combinations, desc="GridSearch", unit="combo")

    grid_start = time.time()

    for params in bar:
        # Compute a stable signature to detect already-done combinations
        signature = str(sorted(params.items()))

        # Skip combinations already present in the checkpoint
        if signature in done_signatures:
            continue

        bar.set_postfix_str("training...")
        fold_scores = []
        fold_times = []

        # --------------------------------------------------------------
        # K-fold loop: one fold for validation, the rest for training
        # --------------------------------------------------------------
        for fold, (train_idx, val_idx) in enumerate(splits, start=1):
            # Split indices into train and validation subsets
            X_tr = X_train.iloc[train_idx]
            y_tr = (
                y_train.iloc[train_idx]
                if hasattr(y_train, "iloc")
                else y_train[train_idx]
            )
            X_va = X_train.iloc[val_idx]
            y_va = (
                y_train.iloc[val_idx]
                if hasattr(y_train, "iloc")
                else y_train[val_idx]
            )

            # Clone the pipeline so every fold starts from scratch
            model_fold = clone(pipe).set_params(**params)

            # Time the fit and the prediction separately
            fold_start = time.time()
            model_fold.fit(X_tr, y_tr)
            fit_elapsed = time.time() - fold_start

            y_prob = model_fold.predict_proba(X_va)[:, 1]
            auc = roc_auc_score(y_va, y_prob)

            fold_scores.append(float(auc))
            fold_times.append(float(fit_elapsed))

            # Update progress bar with the current fold and its AUC
            bar.set_postfix_str(
                f"fold {fold}/{cv} AUC={auc:.4f}"
            )

        # --------------------------------------------------------------
        # Aggregate fold results into mean, std and time summaries
        # --------------------------------------------------------------
        mean_auc = float(np.mean(fold_scores))
        std_auc = float(np.std(fold_scores))
        mean_fit_time = float(np.mean(fold_times))
        total_fit_time = float(np.sum(fold_times))

        # Convert tuples to lists for JSON serialization
        params_json = {
            k: list(v) if isinstance(v, tuple) else v
            for k, v in params.items()
        }

        entry = {
            "signature": signature,
            "params": params_json,
            "fold_scores": fold_scores,
            "fold_times": fold_times,
            "mean_auc": mean_auc,
            "std_auc": std_auc,
            "mean_fit_time": mean_fit_time,
            "total_fit_time": total_fit_time,
        }
        data.append(entry)
        payload["results"] = data

        # --------------------------------------------------------------
        # Update metadata with the current total wall time and write it
        # --------------------------------------------------------------
        payload["metadata"]["total_wall_time"] = time.time() - grid_start

        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        bar.set_postfix_str(f"mean AUC={mean_auc:.4f}")

    bar.close()

    total_wall_time = time.time() - grid_start
    logger.info("Total wall time: %.2f min", total_wall_time / 60)

    # ------------------------------------------------------------------
    # Restore tuples that were converted to lists for JSON
    # ------------------------------------------------------------------
    for entry in data:
        hls_key = "mlpclassifier__hidden_layer_sizes"
        if hls_key in entry["params"]:
            entry["params"][hls_key] = tuple(entry["params"][hls_key])

    # ------------------------------------------------------------------
    # Build the results DataFrame sorted by mean AUC
    # ------------------------------------------------------------------
    df_results = (
        pd.DataFrame(data)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )

    best_params = df_results.iloc[0]["params"]
    logger.info("Best AUC: %.4f", df_results.iloc[0]["mean_auc"])
    logger.info("Best params: %s", best_params)

    return df_results

In [8]:
df_sample = get_stratified_sample(con, "train", n=1_000_000)

print(f"Filas: {len(df_sample):,}")
print(f"CTR:   {df_sample['click'].mean():.6f}")

original = con.sql("SELECT AVG(click) AS ctr FROM train").df()["ctr"].iloc[0]
sampled = df_sample["click"].mean()
print(f"CTR original: {original:.6f}")
print(f"CTR muestra:  {sampled:.6f}")
print(f"Diferencia:   {abs(original - sampled):.6f}")

Filas: 1,000,000
CTR:   0.169787
CTR original: 0.169787
CTR muestra:  0.169787
Diferencia:   0.000000


# entrenamiento

In [17]:
X_train.describe()

,day,hour_sin,hour_cos,log_freq_device_id,log_freq_device_ip,device_type,device_conn_type,banner_pos,C1,C14,C15,C16,C17,C18,C19,C20,C21
count,1000000.000000,1.000000e+06,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.00000,1000000.00000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,25.468615,4.291200e-02,-0.197150,0.334158,3.876115,1.015945,0.332051,0.287616,1004.968597,18846.944605,318.908800,60.10642,2113.35089,1.435035,226.716243,53293.894898,83.331670
std,2.963948,7.199672e-01,0.664032,0.941181,2.736515,0.528832,0.855321,0.504917,1.093149,4952.423667,21.620754,47.35920,608.57522,1.326310,350.656340,49951.915773,70.271652
min,21.000000,-1.000000e+00,-1.000000,0.000000,0.693147,0.000000,0.000000,0.000000,1001.000000,375.000000,120.000000,20.00000,112.00000,0.000000,33.000000,-1.000000,1.000000
25%,23.000000,-7.071068e-01,-0.866025,0.000000,1.609438,1.000000,0.000000,0.000000,1005.000000,16920.000000,320.000000,50.00000,1863.00000,0.000000,35.000000,-1.000000,23.000000
50%,26.000000,1.224647e-16,-0.258819,0.000000,3.258097,1.000000,0.000000,0.000000,1005.000000,20346.000000,320.000000,50.00000,2323.00000,2.000000,39.000000,100049.000000,61.000000
75%,28.000000,7.071068e-01,0.500000,0.000000,5.545177,1.000000,0.000000,1.000000,1005.000000,21894.000000,320.000000,50.00000,2526.00000,3.000000,171.000000,100094.000000,101.000000
max,30.000000,1.000000e+00,1.000000,9.613603,11.892478,5.000000,5.000000,7.000000,1012.000000,24050.000000,1024.000000,1024.00000,2758.00000,3.000000,1839.000000,100248.000000,255.000000


In [20]:
X_train = df_sample.drop(columns=["click"])
y_train = df_sample["click"]

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"CTR:     {y_train.mean():.6f}")

X_train: (1000000, 25)
y_train: (1000000,)
CTR:     0.169787


In [27]:
pipe = make_pipeline(
    AvazuPreprocessor(smoothing=10),
    SelectiveScaler(indices=(3, 4)),
    MLPClassifier(
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.2,
        tol=1e-4,
    ),
)

param_grid = {
    "avazupreprocessor__smoothing": [0, 10],
    "mlpclassifier__hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "mlpclassifier__alpha": [0.0001, 0.001, 0.01],
    "mlpclassifier__max_iter": [100, 200, 500]
}

In [28]:
df_results = run_grid_search_manual_kfold(
    pipe=pipe,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    model_path=MODEL_PATH,
    cv=5,
    scoring="roc_auc",
    results_filename="grid_sklearn.json"
)

INFO:__main__:Grid search: 54 combinations, cv=5
GridSearch: 100%|██████████| 54/54 [2:48:49<00:00, 187.58s/combo, mean AUC=0.7448]      
INFO:__main__:Total wall time: 168.82 min
INFO:__main__:Best AUC: 0.7457
INFO:__main__:Best params: {'avazupreprocessor__smoothing': 10, 'mlpclassifier__hidden_layer_sizes': (100, 50), 'mlpclassifier__alpha': 0.001, 'mlpclassifier__max_iter': 200}


In [23]:
# Grid 1: solo smoothing
param_grid_smoothing = {
    "avazupreprocessor__smoothing": [10, 25, 50],
}

df_smoothing = run_grid_search_manual_kfold(
    pipe=pipe,
    param_grid=param_grid_smoothing,
    X_train=X_train,
    y_train=y_train,
    model_path=MODEL_PATH,
    cv=5,
    scoring="roc_auc",
    results_filename="grid_smoothing.json",
)

INFO:__main__:Grid search: 3 combinations, cv=5
GridSearch: 100%|██████████| 3/3 [09:44<00:00, 194.68s/combo, mean AUC=0.7448]    
INFO:__main__:Total wall time: 9.73 min
INFO:__main__:Best AUC: 0.7451
INFO:__main__:Best params: {'avazupreprocessor__smoothing': 10}


In [24]:
df_smoothing

,signature,params,fold_scores,fold_times,mean_auc,std_auc,mean_fit_time,total_fit_time
0,"[('avazupreprocessor__smoothing', 10)]",{'avazupreprocessor__smoothing': 10},"[0.7458849486374444, 0.7420630249475789, 0.746...","[33.71923470497131, 31.396636962890625, 46.703...",0.745113,0.001573,34.463624,172.318119
1,"[('avazupreprocessor__smoothing', 50)]",{'avazupreprocessor__smoothing': 50},"[0.7442621701308318, 0.7430093077879544, 0.746...","[40.15484285354614, 47.43572187423706, 45.4763...",0.744789,0.001152,45.945155,229.725777
2,"[('avazupreprocessor__smoothing', 25)]",{'avazupreprocessor__smoothing': 25},"[0.7452793015302824, 0.7414423821559322, 0.746...","[42.46008563041687, 31.50644850730896, 32.1480...",0.744712,0.001891,33.842190,169.210948
